# Linear Probing Shuffled-Ident Baseline

This notebook builds a random baseline by shuffling the `ident -> target` mapping,
then runs linear probes on that shuffled target assignment.


In [ ]:
import os
from pathlib import Path
import pandas as pd

from prob.paths_and_io import get_project_root, load_X_from_pt
from prob.prob_config import get_ds_load_config
from prob.prob_models_and_run import linear_models_shuffled_ident_baseline


In [ ]:
project_root = get_project_root()
os.environ["HOME_PROJ_DIR"] = str(project_root)
print("Project root:", project_root)

# ---- Edit these only ----
TARGET_FILE = "weighted_hb_score.pt"
LAYER_NUM = 1
RANDOM_STATE = 96
BASELINE_TAG = "shuffled_ident"

# Cluster-safe CPU budget
CPU_BUDGET = int(
    os.getenv("SLURM_CPUS_PER_TASK")
    or os.getenv("NSLOTS")
    or os.getenv("OMP_NUM_THREADS")
    or 1
)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
N_JOBS = CPU_BUDGET
print(f"CPU_BUDGET={CPU_BUDGET}, N_JOBS={N_JOBS}")

# Optional: re-use best params for faster reruns.
REUSE_BEST_PARAMS = True
BEST_PARAMS_CACHE_DIR = project_root / "data" / "probing" / "_shared_best_params"


In [ ]:
prob_config = get_ds_load_config(target_file=TARGET_FILE)
X = load_X_from_pt(prob_config.output_dir, layer_num=LAYER_NUM)
prob_config.update({"layer_num": LAYER_NUM}, allow_duplicates=True)

print("X shape:", X.shape)
print("target file:", TARGET_FILE)
print("layer:", LAYER_NUM)


In [ ]:
runs = linear_models_shuffled_ident_baseline(
    prob_config,
    X,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    baseline_tag=BASELINE_TAG,
    target_file=TARGET_FILE,
    reuse_best_params=REUSE_BEST_PARAMS,
    best_params_cache_dir=BEST_PARAMS_CACHE_DIR,
)

baseline_df = pd.DataFrame(runs).sort_values(["r2", "rmse"], ascending=[False, True]).reset_index(drop=True)
baseline_df


In [ ]:
display(baseline_df[["experiment", "layer", "r2", "rmse", "mae", "fit_seconds"]])
best = baseline_df.iloc[0]
print(f"Best shuffled-ident baseline: {best['experiment']} | R2={best['r2']:.4f} | RMSE={best['rmse']:.4f}")


In [ ]:
baseline_target_name = f"{Path(TARGET_FILE).stem}_{BASELINE_TAG}"
out_csv = Path(prob_config.output_dir) / baseline_target_name / "experiments" / f"layer_{LAYER_NUM}_linear_baseline_summary.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
baseline_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
